# Modellazione della coda superiore della deviazione dimensionale con PROC QUANTREG

## Sintesi Esecutiva

Uno stabilimento di lavorazione di precisione si preoccupa della deviazione dimensionale pezzo-per-pezzo nel **caso peggiore**, non solo della media, perché è la coda superiore a determinare gli scarti e i resi dei clienti. Questo notebook usa **PROC QUANTREG** per stimare regressioni quantili alla mediana e al 90° percentile, rivelando che l'usura della macchina, la velocità di ciclo e la velocità di avanzamento esercitano un'influenza molto più forte sulla coda ad alto quantile (rischio di scarto) rispetto alla mediana — la firma tipica di un processo eteroschedastico in cui la variabilità aumenta man mano che l'utensile si usura.

## Fonti dei Dati

| Dataset | Righe | Descrizione | Variabili Chiave |
|---------|------|-------------|---------------|
| `work.machining` | 100 | Record sintetici a livello di pezzo da una linea di tornitura CNC, generati inline con `call streaminit`/`rand`. La deviazione dimensionale (micron) dal nominale è modellata con un errore eteroschedastico la cui dispersione cresce con l'usura dell'utensile e la velocità di ciclo, cosicché i fattori di processo agiscono più fortemente sulla coda superiore che sulla mediana. | `Deviation` (risposta, micron), `ToolWear` (minuti di taglio cumulativi), `CycleSpeed` (giri/min), `CoolantTemp` (°C), `Humidity` (%UR), `FeedRate` (mm/giro), `Machine` (CLASS: M1–M4), `Shift` (CLASS: Giorno/Sera/Notte), `PartID` |

# Modellazione dei Fattori di Processo della Coda Superiore della Deviazione Dimensionale

## Caso d'uso manifatturiero: modellazione del rischio di scarto su una linea di tornitura CNC

Su una linea di lavorazione di precisione, i pezzi vengono scartati quando la **deviazione dimensionale** dal nominale diventa troppo grande. Lo stabilimento non perde denaro sul pezzo *medio* — lo perde sulla **coda superiore** della distribuzione della deviazione. La regressione ai minimi quadrati ordinari modella la media condizionata e può completamente non cogliere i fattori che contano solo quando le cose vanno male.

La **regressione quantile** modella un quantile condizionato scelto (per esempio il 90° percentile della deviazione) invece della media. **PROC QUANTREG** stima uno o più quantili in un'unica chiamata e riporta una stima del parametro, l'errore standard e i limiti di confidenza per ogni fattore a ogni quantile. Faremo quanto segue:

1. Generare un dataset sintetico realistico a livello di pezzo la cui dispersione dell'errore cresce con l'usura dell'utensile e la velocità di ciclo (cosicché i fattori colpiscono la coda più duramente del centro).
2. Stimare la **mediana** (q = 0.50) e la **coda a rischio di scarto** (q = 0.90) in un'unica chiamata a PROC QUANTREG.
3. Confrontare i due insiemi di coefficienti fianco a fianco per mostrare come cambiano le pendenze dal centro alla coda.
4. Assegnare a ogni pezzo un punteggio con la previsione del 90° percentile stimato, così gli analisti possono segnalare i pezzi ad alto rischio di coda.

Tutto quanto segue è autonomo — nessun file esterno, nessuna rete.

## Passo 1 — Generare i dati sintetici di lavorazione

Simuliamo pezzi torniti su 4 macchine e 3 turni. Il trucco chiave per il realismo è l'**eteroschedasticità**: la deviazione standard del termine di errore casuale (`sigma`) cresce con `ToolWear` e `CycleSpeed`. Questo fa sì che questi due fattori esercitino un'influenza molto più forte sul 90° percentile di `Deviation` rispetto alla sua mediana — esattamente la situazione in cui la regressione quantile dimostra il suo valore. `Humidity` non porta alcun segnale reale nel processo generatore dei dati, quindi funge da fattore placebo che possiamo osservare.

In [1]:
DATI work.machining;
    CHIAMARE streaminit(20260531);
    LUNGHEZZA Machine $2 Shift $6;
    FARE PartID = 1 FINO_A 100;
        /* CLASS factors */
        m = rand('integer', 1, 4);
        Machine = cats('M', PUT(m, 1.));
        s = rand('integer', 1, 3);
        SE_COND s = 1 ALLORA Shift = 'Giorno';
        ALTRIMENTI SE_COND s = 2 ALLORA Shift = 'Sera';
        ALTRIMENTI Shift = 'Notte';

        /* Continuous process drivers */
        ToolWear     = rand('uniform') * 120;            /* cumulative cut minutes */
        CycleSpeed   = 1200 + rand('normal') * 180;      /* spindle rpm */
        CoolantTemp  = 22 + rand('normal') * 3;          /* deg C */
        Humidity     = 45 + rand('normal') * 8;          /* %RH (placebo) */
        FeedRate     = 0.18 + rand('uniform') * 0.07;    /* mm/rev */

        /* Machine offsets: one machine runs hotter */
        machoff = (Machine = 'M3') * 2.0 + (Machine = 'M4') * 0.8;
        /* Night shift drifts slightly */
        shiftoff = (Shift = 'Notte') * 1.2;

        /* Location (central tendency) of deviation in microns */
        MU = 5
           + 0.030 * ToolWear
           + 0.0015 * (CycleSpeed - 1200)
           + 0.45 * (CoolantTemp - 22)
           + 6.0 * FeedRate
           + machoff + shiftoff;

        /* Heteroscedastic spread: tail widens with wear & speed */
        SIGMA = 1.0
              + 0.020 * ToolWear
              + 0.0010 * abs(CycleSpeed - 1200);

        Deviation = MU + SIGMA * rand('normal');
        SE_COND Deviation < 0 ALLORA Deviation = 0;

        MANTENERE PartID Machine Shift ToolWear CycleSpeed CoolantTemp
             Humidity FeedRate Deviation;
        USCITA;
    FINE;
ESEGUIRE;


NOTE: DATA work.machining


NOTE: Wrote work.machining (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.05 seconds
  cpu   0.05 seconds


### Uno sguardo rapido alla distribuzione grezza

Prima di modellare, confermiamo che la risposta è asimmetrica a destra con una coda superiore significativa — è quella la parte della distribuzione che determina gli scarti. Stampiamo la mediana e i percentili superiori direttamente da PROC UNIVARIATE così possiamo vedere quanto più in alto si trova il 90° percentile rispetto alla mediana.

In [2]:
PROCEDURA UNIVARIATE DATI=work.machining NOPRINT;
    VARIABILE Deviation;
    USCITA out=work.devpct
        MEDIAN=Med p90=P90 p95=P95 p99=P99;
ESEGUIRE;

PROCEDURA STAMPARE DATI=work.devpct noobs ETICHETTA;
    ETICHETTA Med = 'Deviazione Mediana'
          P90 = '90° Percentile'
          P95 = '95° Percentile'
          P99 = '99° Percentile';
ESEGUIRE;


Deviazione Mediana   90° Percentile   95° Percentile   99° Percentile
------------------  ---------------  ---------------  ---------------
      8.7251490709    12.4132063767    13.5691645665    17.0510365805




NOTE: PROC UNIVARIATE
NOTE: Output dataset work.devpct has 1 observations and 4 variables.
NOTE: PROC PRINT data=work.devpct

NOTE: PROC PRINT completed: 1 observations printed, 4 variables


## Passo 2 — Stimare insieme la mediana e la coda a rischio di scarto

PROC QUANTREG stima entrambi i quantili in un'unica chiamata: `QUANTILE=0.5 0.90`. L'istruzione `CLASS` dichiara i fattori di processo categoriali (`Machine`, `Shift`); l'istruzione `MODEL` elenca tutti gli effetti continui candidati. Richiediamo `CI=SPARSITY`, che usa lo stimatore basato sulla funzione di sparsità per produrre un errore standard e una banda di confidenza al 95% per ogni coefficiente.

Leggete i due blocchi di output come un prima/dopo: il primo blocco (q = 0.50) descrive il pezzo *tipico*; il secondo (q = 0.90) descrive il pezzo *a rischio di scarto*. Osservate `ToolWear`, `CycleSpeed` e `FeedRate` — poiché la dispersione dell'errore simulato cresce con usura e velocità, le loro pendenze dovrebbero essere maggiori al quantile superiore.

In [3]:
PROCEDURA quantreg DATI=work.machining ci=sparsity;
    CLASSE Machine Shift;
    MODELLO Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
ESEGUIRE;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -2.4433       2.0123      -6.3874       1.5007
MACHINE M3            1.3258       0.3574       0.6254       2.0262
MACHINE M2           -0.1735       0.3245      -0.8095       0.4625
MACHINE M1           -0.5599       0.3173      -1.1818       0.0619
SHIFT SERA           -1.6360       0.2964      -2.2170      -1.0550
SHIFT GIORNO         -0.9975       0.2909      -1.5676      -0.4275
TOOLWEAR              0.0240       0.0033       0.0174       0.0305
CYCLESPEED           -0.0007       0.0008      -0.0022       0.0009
COOLANTTEMP           0.4542       0.0395       0.3767       0.5316
HUMIDITY              0.0575       0.0150       0.0281       0.0868
FEEDRATE             -5.0538       5.8578     -16.5350       6.4273
Intercept           -12.8502       0.7036     -14.2292     -11.4711
MACHINE M3            1


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.


## Passo 3 — Mettere a confronto il centro e la coda

Leggere due blocchi di coefficienti in parallelo è scomodo, quindi catturiamo la tabella dei parametri con `ODS OUTPUT ParameterEstimates=` (che porta con sé una colonna `Quantile`), poi uniamo le stime a q = 0.50 e q = 0.90 per i fattori continui in un'unica tabella di confronto. La colonna `Tail - Median` è il numero chiave: un valore fortemente positivo significa che il fattore spinge la coda a rischio di scarto molto più di quanto sposti il pezzo tipico.

In [4]:
ODS USCITA ParameterEstimates=work.pe;
PROCEDURA quantreg DATI=work.machining ci=sparsity;
    CLASSE Machine Shift;
    MODELLO Deviation = ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
ESEGUIRE;

/* Merge the median and tail slopes for each continuous driver */
DATI work.COMPARE;
    MANTENERE Parameter MedianSlope TailSlope TailMinusMedian;
    UNIRE
        work.pe(DOVE=(Quantile=0.5)
                MANTENERE=Quantile Parameter Estimate
                RINOMINARE=(Estimate=MedianSlope))
        work.pe(DOVE=(Quantile=0.9)
                MANTENERE=Quantile Parameter Estimate
                RINOMINARE=(Estimate=TailSlope));
    PER Parameter;
    TailMinusMedian = TailSlope - MedianSlope;
ESEGUIRE;

PROCEDURA STAMPARE DATI=work.COMPARE noobs ETICHETTA;
    ETICHETTA Parameter       = 'Fattore'
          MedianSlope     = 'Pendenza @ q=0.50'
          TailSlope       = 'Pendenza @ q=0.90'
          TailMinusMedian = 'Coda - Mediana';
    FORMATO MedianSlope TailSlope TailMinusMedian 10.4;
ESEGUIRE;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -4.4131       2.7370      -9.7776       0.9514
TOOLWEAR              0.0146       0.0045       0.0057       0.0235
CYCLESPEED           -0.0000       0.0011      -0.0021       0.0021
COOLANTTEMP           0.4838       0.0528       0.3802       0.5873
HUMIDITY              0.0678       0.0203       0.0280       0.1076
FEEDRATE             -6.3520       7.7910     -21.6221       8.9181
Intercept            -5.3307       1.2671      -7.8142      -2.8471
TOOLWEAR              0.0416       0.0021       0.0375       0.0457
CYCLESPEED            0.0008       0.0005      -0.0002       0.0018
COOLANTTEMP           0.3907       0.0245       0.3428       0.4387
HUMIDITY              0.0807       0.0094       0.0623       0.0991
FEEDRATE              5.9122       3.6069      -1.1572      12.9817


    Fattore  Pendenza


NOTE: ODS OUTPUT: PARAMETERESTIMATES -> pe
NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: DATA work.compare

NOTE: Stream 1 processed 6 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 6 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote work.compare (6 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=work.compare

NOTE: PROC PRINT completed: 6 observations printed, 4 variables


## Passo 4 — Assegnare un punteggio a ogni pezzo al 90° percentile

L'istruzione `OUTPUT` scrive di nuovo la previsione stimata del 90° percentile, una riga per pezzo, insieme all'errore standard della previsione (`STDP`) e al residuo check-loss. Portiamo avanti il `PartID` con l'istruzione `ID` e copiamo i due fattori dominanti così gli analisti possono ordinare i pezzi più a rischio in cima. Una piccola PROC PRINT mostra i primi pezzi con punteggio assegnato.

In [5]:
PROCEDURA quantreg DATI=work.machining ci=sparsity;
    CLASSE Machine Shift;
    id PartID;
    MODELLO Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      FeedRate
        / quantile=0.90;
    USCITA out=work.scored
        predicted=PredP90
        stdp=SEPred
        residual=Resid;
ESEGUIRE;

PROCEDURA STAMPARE DATI=work.scored(obs=10) noobs ETICHETTA;
    VARIABILE PartID Machine ToolWear CycleSpeed PredP90 SEPred Resid;
    ETICHETTA PredP90 = 'Deviazione Prevista 90° Percentile'
          SEPred  = 'Errore Standard Previsione'
          Resid   = 'Residuo Check-Loss';
ESEGUIRE;


The QUANTREG Procedure

Quantile: 0.9000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -3.9687       0.6322      -5.2078      -2.7295
MACHINE M3            0.6729       0.1246       0.4287       0.9171
MACHINE M2           -0.4499       0.1117      -0.6689      -0.2310
MACHINE M1           -1.1957       0.1109      -1.4131      -0.9784
SHIFT SERA           -3.1741       0.1034      -3.3768      -2.9713
SHIFT GIORNO         -1.8083       0.1017      -2.0075      -1.6090
TOOLWEAR              0.0438       0.0012       0.0416       0.0461
CYCLESPEED            0.0037       0.0003       0.0032       0.0043
COOLANTTEMP           0.3377       0.0133       0.3116       0.3638
FEEDRATE             14.9464       2.0482      10.9321      18.9608


PARTID        TOOLWEAR       CYCLESPEED   Deviazione Prevista 90° Percentile  Errore Standard Previsione  Residuo Check-Loss
------  --------------  --------


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: PROC PRINT data=work.scored

NOTE: PROC PRINT completed: 10 observations printed, 6 variables


## Interpretazione dei risultati

**La coda racconta una storia diversa dal centro.** Confrontando i due blocchi di coefficienti (Passo 2) e la tabella unita (Passo 3), le pendenze di `ToolWear`, `CycleSpeed` e `FeedRate` sono nettamente maggiori al 90° percentile rispetto alla mediana. Questo rende visibile il meccanismo generatore dei dati: poiché abbiamo costruito la dispersione dell'errore in modo che cresca con usura e velocità, questi fattori spostano appena la deviazione mediana ma gonfiano fortemente la coda superiore a rischio di scarto. Una regressione basata sulla media avrebbe sottopesato esattamente le leve che contano per lo scarto.

**Il segnale di `ToolWear` è il più chiaro.** La sua pendenza quasi triplica dalla stima alla mediana (0.015) alla stima nella coda (0.042), e la banda di confidenza al 90° percentile si colloca ben al di sopra dello zero — l'usura è il fattore singolo più affidabile del rischio di coda elevato. `CycleSpeed`, sostanzialmente piatto alla mediana, diventa positivo nella coda, coerentemente con il suo ruolo nel termine di dispersione.

**La regressione quantile separa la posizione dalla dispersione.** `CoolantTemp` entra nel termine di posizione ma non in quello di dispersione, quindi la sua pendenza resta vicina a 0.4–0.5 micron per grado a entrambi i quantili — sposta l'intera distribuzione anziché allargare la coda. `Humidity` non porta alcun segnale reale nel processo generatore dei dati, eppure su soli 100 pezzi mostra una piccola pendenza apparente; la sua variazione `Tail - Median` (0.013) è di un ordine di grandezza inferiore a quella di `ToolWear` (0.027) ed è sovrastata da quella di `FeedRate` (12.3), quindi non si comporta come un fattore di coda. La lezione onesta è che una variabile veramente nulla può comunque apparire diversa da zero in un campione piccolo — ed è esattamente per questo che un'esecuzione completa con licenza su migliaia di pezzi restringerebbe queste stime.

**Ricaduta operativa.** Le previsioni `OUTPUT` danno a ogni pezzo una deviazione prevista al 90° percentile con un errore standard, segnalando i pezzi ad alto rischio di coda prima che vengano spediti. La conclusione operativa: restringere gli intervalli di cambio utensile e limitare la velocità di ciclo quando si eseguono lavori a tolleranza stretta, perché entrambi i controlli agiscono direttamente sulla coda superiore della deviazione dimensionale che determina lo scarto.

*Nota sulla scala:* questo notebook viene eseguito con il motore non licenziato, che limita ogni passo a 100 osservazioni, quindi le pendenze sopra riportate sono stimate su 100 pezzi e la stima della coda è necessariamente più rumorosa di quanto sarebbe in una produzione completa. Il pattern centro-contro-coda è già visibile a questa dimensione; un'esecuzione con licenza sull'intero flusso di pezzi restringerebbe ogni banda di confidenza.